# Descriptive statistics and bivariate tests

This notebook reads the cleaned crash dataset produced by
[`loading_and_cleaning_data.ipynb`](loading_and_cleaning_data.ipynb), applies **the same
filters as `modeling_development.ipynb`** so that the descriptive tables and the
estimated models describe one and the same sample, and produces:

1. A summary-statistics table for one-hot-encoded categorical variables and a few
   continuous infrastructure variables.
2. A pivot table of counts and percentages by injury severity, together with
   bivariate tests (Chi-square for categorical variables; ANOVA or Kruskal-Wallis
   for continuous variables, depending on the normality test).

The two output tables are saved as CSV files and displayed inline at the end of
their respective sections so the analysis can be inspected directly in the notebook.


In [1]:
# All imports for this notebook (kept together at the top, as required).
import warnings
warnings.filterwarnings('ignore')

import re

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, kruskal, kstest

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)


## 1. Load and filter the dataset

The three filters below are those of `modeling_development.ipynb`, applied in the
same order, so that the two notebooks describe the same population:

1. `severity != -1` -- the severity is unknown;
2. `Vehicle` in E-bike / Bike / E-PMD / **Pedestrian** -- pedestrians are part of
   the estimation sample, they are the third party of one of the four segments;
3. `dropna(['age', 'severity'])` -- the age is needed by every model;
4. the union of the four estimation segments, which removes the 200 rows that
   belong to no model: 199 with `catu == 3` -- pedestrians, excluded by the
   `catu` filter of the car, MMV and single-vehicle models -- and one fatal
   micro-mobility crash removed by the `severity != 3` filter of the MMV model.

The tables therefore describe **exactly** the 15 386 observations the four models
are estimated on.


In [2]:
# low_memory=False removes the DtypeWarning caused by mixed-type columns in the
# combined CSV.
dataset = pd.read_csv('final_processed_crash_dataset.csv', low_memory=False)

dataset = dataset.loc[dataset['severity'] != -1]
dataset = dataset.loc[dataset['Vehicle'].isin(['E-PMD', 'Bike', 'E-bike', 'Pedestrian'])]

# Same drop as `modeling_development.ipynb`: an observation without age enters no
# model, so it must not enter the descriptive tables either.
dataset = dataset.dropna(subset=['age', 'severity'])

# The four estimation segments of `modeling_development.ipynb`, reproduced here so
# that the descriptive tables cover exactly the observations the models use.
# `catu` and `severity != 3` cannot be applied globally: the pedestrian segment
# keeps its `catu == 3` rows -- they are the pedestrians -- and only the MMV
# segment drops the fatalities. Taking the union of the four masks is the only
# way to reproduce the estimation sample without distorting any segment.
second_party = dataset['vehicle_type_2']
is_driver_or_passenger = dataset['catu'].isin([1, 2])

is_car = second_party.isin(['Cars', 'Large motorized vehicle',
                            'Light motorized vehicle']) & is_driver_or_passenger
is_mmv = ((second_party == 'Micromobility vehicle') & is_driver_or_passenger
          & (dataset['severity'] != 3))
is_sv = (second_party == 'No other vehicle') & is_driver_or_passenger
is_pedestrian = dataset['Num_Acc'].isin(
    dataset.loc[second_party == 'Pedestrian', 'Num_Acc']
)

dataset = dataset.loc[is_car | is_mmv | is_pedestrian | is_sv]

# Convenience alias used in the bivariate tables further down.
dataset['sev'] = dataset['severity']

print(f'Rows after filtering: {len(dataset):,}')   # doit valoir 15 386
print(f'Unique accidents:     {dataset["Num_Acc"].nunique():,}')
dataset.head()


Rows after filtering: 15,386
Unique accidents:     13,024


,Unnamed: 0,Num_Acc,day,month,Year,hrmn,lum,com,int,atm,col,adr,lat,long,geometry,catr,circ,nbv,vosp,prof,plan,surf,infra,situ,vma,...,Gender_driver,severity_driver,Maneuver,Obstacle,Vehicle type,danger_rank,age_2,vehicle_type_2,Vehicle_2,Maneuver_2,Gender_2,severity_2,Point of impact_2,id_vehicule_opposite_2,Age category involved,age_3,vehicle_type_3,Vehicle_3,Maneuver_3,Gender_3,severity_3,Point of impact_3,id_vehicule_opposite_3,age_opposite_mean,sev
1,1,2.022000e+11,21,10,2022,16:32,1,75106,1,1,3,RUE DE VAUGIRARD,48.847999,2.330176,POINT (2.330176 48.847999),4,2,2,0,1,1,1,0,1,30,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,62.0,Cars,Car,Door openied,Female,1.0,Front,813 926,61+,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,2
3,3,2.022000e+11,20,10,2022,13:00,1,75105,2,1,3,BOULEVARD SAINT GERMAIN,48.851387,2.343186,POINT (2.343186 48.851387),4,1,3,1,1,1,1,0,5,30,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,20.0,Cars,Car,Turning right,Male,1.0,Front,813 924,21-40,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,2
4,4,2.022000e+11,21,10,2022,11:25,1,75113,4,1,6,BOULEVARD KELLERMANN,48.821028,2.354515,POINT (2.354515 48.821028),4,2,6,1,2,2,2,0,5,50,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,999.0,No other vehicle,NaN,Missing,NaN,NaN,NaN,NaN,NaN,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2
7,7,2.022000e+11,21,10,2022,19:40,5,93049,2,1,3,Avenue Jean Jaurès / Avenue Victor Hugo,48.858700,2.506540,POINT (2.50654 48.8587),4,1,2,0,1,1,1,0,1,30,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,48.0,Cars,Car,Without change of direction,Female,1.0,Front,813 884,41-60,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.0,2
9,9,2.022000e+11,21,10,2022,20:00,5,92048,1,1,2,JEAN JAURES (AVENUE) N° 8 A 72,48.812880,2.246200,POINT (2.2462 48.81288),4,2,2,0,4,1,1,0,1,50,...,999,999.0,Without change of direction,Other,Micromobility vehicle,4.0,62.0,Cars,Car,Other,Male,1.0,Back,813 866,61+,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,2


## 2. Variable groups

We work with two units of observation:

- **Accident-level frame** (`data_acc`): one row per accident, used for variables
  that describe the crash environment (lighting, weather, road type, ...).
- **User-level frame** (`dataset`): one row per person involved, used for
  individual variables (age, gender, helmet, ...).

`var_for_acc` and `var_cont` list the variables that must be analysed at the
accident level and the continuous variables, respectively.


In [3]:
data_acc = dataset.drop_duplicates('Num_Acc')

# Trois familles de variables, et une seule regle : tout ce qui n'est pas propre
# a l'individu ni au tiers decrit l'accident, et se compte donc une fois par
# accident (`var_for_acc`, construit en section 4 par complement).

# Propre a l'usager observe : une ligne = une personne.
RIDER_VARS = {
    'age', 'Age category', 'Gender', 'Vehicle', 'User category', 'Helmet',
    'Reflective jacket', 'Trip purpose', 'Number of passengers',
    'Maneuver', 'Point of impact',
}

# Propre aux tiers : lues en « au moins un tiers presente cette
# caracteristique », en combinant les colonnes `_2` (premier tiers) et `_3`
# (second). Voir SECOND_PARTY_GROUPS en section 3.
SECOND_PARTY_VARS = {
    'vehicle_type_2', 'vehicle_type_3', 'Vehicle_2', 'Vehicle_3',
    'Maneuver_2', 'Maneuver_3', 'Gender_2', 'Gender_3',
    'Point of impact_2', 'Point of impact_3',
    'age_2', 'age_opposite_mean', 'Age category involved',
}

# Continuous variables analysed in the bivariate section.
var_cont = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]

## 3. Summary statistics for one-hot-encoded categorical and continuous variables

We one-hot-encode a small set of categorical variables, merge in the continuous
infrastructure variables from the accident-level frame, and compute mean / median /
min / max for each column. The resulting table is saved as
`summary_statistics_categorical_continuous.csv` and is displayed below.


In [4]:
var_added = pd.get_dummies(
    dataset[['Num_Acc', 'Cycle facilities', 'Pavement',
             'Crossroad', 'positionnement_piste',
             'Road type', 'Reglementation']]
)
var_added = var_added.astype(int)  # boolean -> integer

var_added = var_added.merge(
    data_acc[['Num_Acc', 'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit']],
    on='Num_Acc',
    how='left'
)

results_df = pd.DataFrame([
    {
        'Column': col,
        'Mean':   var_added[col].mean(),
        'Median': var_added[col].median(),
        'Min':    var_added[col].min(),
        'Max':    var_added[col].max(),
    }
    for col in var_added.columns
])

results_df.to_csv('summary_statistics_categorical_continuous.csv', index=False)
results_df


,Column,Mean,Median,Min,Max
0,Num_Acc,2.021233e+11,2.021000e+11,2.019000e+11,2.023001e+11
1,Cycle facilities_Bus lane,1.197842e-01,0.000000e+00,0.000000e+00,1.000000e+00
2,Cycle facilities_Cycle lane,1.464318e-01,0.000000e+00,0.000000e+00,1.000000e+00
3,Cycle facilities_Cycle path/Greenway,1.767191e-01,0.000000e+00,0.000000e+00,1.000000e+00
4,Cycle facilities_No cycle facilities,5.257377e-01,1.000000e+00,0.000000e+00,1.000000e+00
5,Cycle facilities_Pedestrianized street,3.132718e-02,0.000000e+00,0.000000e+00,1.000000e+00
6,Pavement_Asphalt,7.528272e-01,1.000000e+00,0.000000e+00,1.000000e+00
7,Pavement_Concrete,8.904199e-03,0.000000e+00,0.000000e+00,1.000000e+00
8,Pavement_Other,1.827636e-01,0.000000e+00,0.000000e+00,1.000000e+00
9,Pavement_Paved,5.550500e-02,0.000000e+00,0.000000e+00,1.000000e+00


## 4. Bivariate analysis (counts, percentages, p-values)

For each categorical variable we report counts and percentages by severity level,
and a Chi-square p-value per category (NaN when expected frequencies are below 5,
to respect Cochran's rule). For each continuous variable we report the median and
interquartile range by severity, and an ANOVA p-value when the groups are normal
(Kolmogorov-Smirnov test) or a Kruskal-Wallis p-value otherwise.


In [5]:
def calculate_chi2_p_value_for_each_category(data, categorical_var):
    """Per-category Chi-square p-value, NaN if Cochran's rule is violated."""
    p_values = {}
    for category in data[categorical_var].unique():
        contingency = pd.crosstab(data['severity'], data[categorical_var] == category)
        chi2, p, _, expected = chi2_contingency(contingency)
        if np.any(expected < 5):
            p_values[category] = np.nan
        else:
            p_values[category] = '<0.001' if p < 0.001 else str(round(p, 3))
    return p_values


def count_by_severity(data, categorical_var):
    """Counts and percentages of `categorical_var` within each severity level."""
    counts = (data
              .groupby(['severity', categorical_var])
              .size()
              .astype(int)
              .reset_index(name='count'))
    total_counts = counts.groupby(categorical_var)['count'].transform('sum')
    counts['percentage'] = (counts['count'] / total_counts * 100).round(2)
    counts['count_percentage'] = counts.apply(
        lambda row: f"{row['count']} ({row['percentage']}%)", axis=1
    )
    return counts


def compare_continuous_variable(data, group_var, continuous_var):
    """ANOVA if all groups are normal (KS test, alpha=0.05), Kruskal-Wallis otherwise."""
    groups_data = [
        data[data[group_var] == g][continuous_var].dropna().values
        for g in data[group_var].unique()
    ]
    normal = all(
        kstest(g, 'norm', args=(g.mean(), g.std())).pvalue > 0.05
        for g in groups_data
    )
    _, p_value = (f_oneway(*groups_data) if normal else kruskal(*groups_data))
    return '<0.001' if p_value < 0.001 else str(round(p_value, 3))


# --- Caracteristiques du tiers : « au moins un » ------------------------------
# Un usager peut faire face a deux tiers (`_2` et `_3`). Compter une modalite par
# tiers gonflerait les effectifs et donnerait 2 la ou les deux tiers partagent la
# caracteristique : on lit donc « au moins un tiers presente cette
# caracteristique ». Consequence : les pourcentages d'une meme variable ne
# somment plus a 100 %.
SECOND_PARTY_GROUPS = {
    'vehicle_type_2': ['vehicle_type_2', 'vehicle_type_3'],
    'Vehicle_2': ['Vehicle_2', 'Vehicle_3'],
    'Maneuver_2': ['Maneuver_2', 'Maneuver_3'],
    'Gender_2': ['Gender_2', 'Gender_3'],
    'Point of impact_2': ['Point of impact_2', 'Point of impact_3'],
}

SEVERITY_LEVELS = [1, 2, 3]


def any_party_levels(data, columns):
    """Modalites presentes chez l'un ou l'autre des tiers."""
    levels = set()
    for column in columns:
        if column in data.columns:
            levels.update(data[column].dropna().unique())
    return sorted(levels, key=str)


def any_party_mask(data, columns, level):
    """Individus dont AU MOINS un tiers presente la modalite."""
    mask = pd.Series(False, index=data.index)
    for column in columns:
        if column in data.columns:
            mask |= (data[column] == level)
    return mask


def chi2_p_value(severity, mask):
    """Chi-2 de la modalite contre la severite ; None si Cochran est violee."""
    contingency = pd.crosstab(severity, mask)
    if contingency.shape[1] < 2 or contingency.shape[0] < 2:
        return None
    _, p_value, _, expected = chi2_contingency(contingency)
    if np.any(expected < 5):
        return None
    return '<0.001' if p_value < 0.001 else str(round(p_value, 3))


def any_party_block(sample, variable, columns):
    """Bloc « au moins un tiers » : effectifs par severite et test par modalite."""
    records = []
    for level in any_party_levels(sample, columns):
        mask = any_party_mask(sample, columns, level)
        total = int(mask.sum())
        if not total:
            continue
        counts = sample.loc[mask, 'severity'].value_counts()
        record = {'Variable': variable, 'Category': level}
        for severity in SEVERITY_LEVELS:
            count = int(counts.get(severity, 0))
            record[severity] = f'{count} ({round(count / total * 100, 2)}%)'
        record['p_value'] = chi2_p_value(sample['severity'], mask)
        records.append(record)
    return pd.DataFrame.from_records(records) if records else None

In [6]:
categorical_vars = [
    'Age category', 'Gender', 'Vehicle', 'User category', 'Helmet', 'sev',
    'Point of impact_2', 'Reflective jacket', 'Lighting conditions',
    'Weather conditions', 'Point of impact', 'Maneuver', 'Cycle facilities',
    'Accident location', 'Trip purpose', 'Max speed', 'Intersection',
    'Crossroad', 'Long profile', 'Pavement', 'Surface condition', 'Road width',
    'vehicle_type_2', 'Maneuver_2', 'Gender_2', 'Age category involved',
    'positionnement_piste', 'Road type'
]

continuous_vars = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'largeurchaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]

# Tout ce qui n'est ni propre a l'usager ni propre au tiers decrit l'accident.
# `sev` est la variable expliquee, pas un descripteur de l'accident.
var_for_acc = [variable for variable in categorical_vars + continuous_vars
               if variable not in RIDER_VARS
               and variable not in SECOND_PARTY_VARS
               and variable not in ('sev', 'severity')]
print(f'Niveau accident ({len(var_for_acc)}) : {", ".join(var_for_acc)}\n')

pivot_results = []

# Categorical variables: counts/percentages and Chi-square per category.
for var in categorical_vars:
    if var in SECOND_PARTY_GROUPS:
        block = any_party_block(dataset, var, SECOND_PARTY_GROUPS[var])
        if block is not None:
            pivot_results.append(block)
        continue
    df = data_acc if var in var_for_acc else dataset
    counts_df = count_by_severity(df, var)
    pivot_df = counts_df.pivot(index=var, columns='severity',
                               values='count_percentage').reset_index()
    pivot_df['Variable'] = var
    pivot_df['Category'] = pivot_df[var]
    pivot_df.drop(columns=var, inplace=True)
    p_values = calculate_chi2_p_value_for_each_category(df, var)
    pivot_df['p_value'] = pivot_df['Category'].map(p_values)
    pivot_results.append(pivot_df)

# Continuous variables: median [Q1-Q3] and ANOVA / Kruskal-Wallis p-value.
for var in continuous_vars:
    df = data_acc if var in var_for_acc else dataset
    df = df[['severity', var]].dropna(subset=[var])
    df = df[df[var] != 999]  # 999 is the Biogeme missing-data sentinel
    stats_df = df.groupby('severity')[var].agg(
        median='median',
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    ).reset_index()
    stats_df['median_iqr'] = stats_df.apply(
        lambda row: f"{row['median']:.2f} [{row['q1']:.2f}-{row['q3']:.2f}]", axis=1
    )
    stats_df = stats_df[['severity', 'median_iqr']].set_index('severity').T
    stats_df.columns = [1, 2, 3]
    stats_df['Variable'] = var
    stats_df['p_value'] = compare_continuous_variable(df, 'severity', var)
    pivot_results.append(stats_df)

final_pivot_df = pd.concat(pivot_results, ignore_index=True)
cols_order = ['Variable', 'Category'] + [c for c in final_pivot_df.columns
                                          if c not in ['Variable', 'Category']]
final_pivot_df = final_pivot_df[cols_order]

# Rename a few raw column names for the printed/exported table.
final_pivot_df = final_pivot_df.replace({
    'vehicle_type_2': 'Third-party vehicle type',
    'Maneuver_2': 'Third-party maneuver',
    'Gender_2': 'Third-party gender',
    'Point of impact_opposite': 'Third-party impact location',
    'vma': 'Speed limit',
    'surfacechaussee': 'Road surface width',
    'pentemoyenne': 'Average slope',
    'largeurtrottoirdroit': 'Sidewalk width',
    'age': 'Individual age',
    'number of involved vehicles': 'Number of vehicles involved',
})

print(f'Rows in the final pivot table: {len(final_pivot_df)}')
final_pivot_df


Niveau accident (18) : Lighting conditions, Weather conditions, Cycle facilities, Accident location, Max speed, Intersection, Crossroad, Long profile, Pavement, Surface condition, Road width, positionnement_piste, Road type, number of involved vehicles, vma, largeurchaussee, pentemoyenne, largeurtrottoirdroit

Rows in the final pivot table: 141


,Variable,Category,1,2,3,p_value
0,Age category,0-20,344 (16.87%),1687 (82.74%),8 (0.39%),<0.001
1,Age category,21-40,1163 (15.25%),6428 (84.28%),36 (0.47%),<0.001
2,Age category,41-60,398 (9.58%),3728 (89.72%),29 (0.7%),<0.001
3,Age category,61+,75 (4.8%),1464 (93.67%),24 (1.54%),<0.001
4,Gender,Female,364 (7.22%),4652 (92.28%),25 (0.5%),<0.001
5,Gender,Male,1617 (15.63%),8656 (83.67%),72 (0.7%),<0.001
6,Vehicle,Bike,1115 (12.56%),7708 (86.83%),54 (0.61%),0.358
7,Vehicle,E-PMD,569 (14.24%),3402 (85.14%),25 (0.63%),0.011
8,Vehicle,E-bike,160 (14.39%),941 (84.62%),11 (0.99%),0.079
9,Vehicle,Pedestrian,137 (9.78%),1257 (89.72%),7 (0.5%),0.001


## 5. Export the bivariate table


In [7]:
final_pivot_df.to_csv('descriptive_statistics_and_bivariate_tests.csv',
                      index=False, sep=';')
print('Saved to descriptive_statistics_and_bivariate_tests.csv')
final_pivot_df.head(20)


Saved to descriptive_statistics_and_bivariate_tests.csv


,Variable,Category,1,2,3,p_value
0,Age category,0-20,344 (16.87%),1687 (82.74%),8 (0.39%),<0.001
1,Age category,21-40,1163 (15.25%),6428 (84.28%),36 (0.47%),<0.001
2,Age category,41-60,398 (9.58%),3728 (89.72%),29 (0.7%),<0.001
3,Age category,61+,75 (4.8%),1464 (93.67%),24 (1.54%),<0.001
4,Gender,Female,364 (7.22%),4652 (92.28%),25 (0.5%),<0.001
5,Gender,Male,1617 (15.63%),8656 (83.67%),72 (0.7%),<0.001
6,Vehicle,Bike,1115 (12.56%),7708 (86.83%),54 (0.61%),0.358
7,Vehicle,E-PMD,569 (14.24%),3402 (85.14%),25 (0.63%),0.011
8,Vehicle,E-bike,160 (14.39%),941 (84.62%),11 (0.99%),0.079
9,Vehicle,Pedestrian,137 (9.78%),1257 (89.72%),7 (0.5%),0.001


## 6. Export des tableaux LaTeX

`latex_tables.make_latex_tables` transforme `final_pivot_df` en deux `longtable`
(variables catégorielles regroupées par section, puis variables continues).
La structure des sections, les libellés et l'ordre des modalités se modifient
dans `DEFAULT_STRUCTURE` / `DEFAULT_CONTINUOUS` du module `latex_tables.py`.


In [8]:
from pathlib import Path

from latex_tables import make_latex_tables

# Les variables qui decrivent l'accident lui-meme sont comptees une fois par
# accident (voir section 4) : elles portent un asterisque dans les tableaux.
# Les libelles renommes en fin de pipeline doivent aussi le porter, d'ou les
# entrees ajoutees a `var_for_acc`.
CRASH_LEVEL_STARRED = set(var_for_acc) | {'Number of vehicles involved',
                                          'Speed limit', 'Road surface width',
                                          'Average slope', 'Sidewalk width'}
CRASH_LEVEL_NOTE = (r'\footnotesize $^{*}$ Variable describing the crash itself: '
                    r'counted once per accident rather than once per individual.')

tex_cat, tex_cont = make_latex_tables(final_pivot_df,
                                      starred=CRASH_LEVEL_STARRED)

with open(Path('results') / 'tables' / 'table_data.tex', 'w') as f:
    f.write(tex_cat + '\n\n' + CRASH_LEVEL_NOTE + '\n\n'
            + tex_cont + '\n\n' + CRASH_LEVEL_NOTE + '\n')

print('Saved to table_data.tex')
print(tex_cat[:1500])

Saved to table_data.tex
\renewcommand{\arraystretch}{0.95}
\begin{longtable}{
p{4.5cm}
>{\centering\arraybackslash}p{2.5cm}
>{\centering\arraybackslash}p{2.5cm}
>{\centering\arraybackslash}p{2.5cm}
>{\centering\arraybackslash}p{1.3cm}
}
\caption{Descriptive statistics (total and percentage) for each categorical variables and resulting p-value of the Pearson's Chi-square test by level of the category} \label{tab:data_cat} \\
\hline
\textbf{Category} & \textbf{No injury} & \textbf{Injury} & \textbf{Fatality} & \textbf{p-value} \\
\hline
\endfirsthead
\multicolumn{5}{c}{\textit{(continued)}} \\
\hline
\textbf{Category} & \textbf{No injury} & \textbf{Injury} & \textbf{Fatality} & \textbf{p-value} \\
\hline
\endhead
\hline
\multicolumn{5}{r}{\textit{Continued on next page}} \\
\endfoot
\hline
\endlastfoot
\multicolumn{5}{c}{\textbf{Individual and vehicle characteristics}} \\
\hline
\multicolumn{5}{c}{{Gender}} \\
Female & 364 (7.22) & 4652 (92.28) & 25 (0.5) & $<$0.001 \\
Male & 1617 (15.63

## 7. Une table descriptive par segment d'estimation

Les tables ci-dessus portent sur la reunion des quatre segments. Comme les
modeles sont estimes segment par segment, chacun recoit ici la meme table sur
ses propres lignes : memes variables, memes effectifs et pourcentages par
niveau de severite, memes tests bivaries. Les fichiers sont ecrits dans
`results/tables/`.


In [9]:
# --- One descriptive table per estimation segment -----------------------------
# The table above pools the four segments. The models, however, are estimated
# segment by segment, so each of them deserves the same table on its own rows:
# same variables, same counts and percentages by severity, same bivariate tests.
#
# Variables describing the CRASH itself (`var_for_acc`) are counted once per
# accident, the others once per individual: an accident involving three people
# would otherwise weigh three times in the lighting or the road-width
# distribution. Those variables carry an asterisk in the LaTeX tables.
#
# Second-party characteristics come in two copies -- `_2` for the first other
# party, `_3` for the second. They are read as "at least one of the other
# parties has this characteristic", so an individual facing two parties counts
# once per characteristic present, and the percentages of one variable may add
# up to more than 100%.
#
# Two differences with the pooled table, both forced by the data:
#   - the MMV segment has no fatality (its single fatal observation is dropped
#     before estimation), so its severity-3 column stays empty;
#   - a segment being smaller, Cochran's rule is violated far more often, and
#     the corresponding p-values are left empty rather than reported.

from pathlib import Path

from latex_tables import make_categorical_table, make_continuous_table

SEGMENT_TABLES_DIRECTORY = Path('results') / 'tables'
SEGMENT_TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)

SEVERITY_LEVELS = [1, 2, 3]

# `CRASH_LEVEL_STARRED` et `CRASH_LEVEL_NOTE` viennent de la section 6.

# SECOND_PARTY_GROUPS, any_party_levels/mask et any_party_block viennent de
# la section 4 ; SEVERITY_LEVELS aussi.


def segment_masks(data):
    """Les quatre segments d'estimation, reproduits sur `data`."""
    second_party = data['vehicle_type_2']
    is_driver_or_passenger = data['catu'].isin([1, 2])
    return {
        'Car crashes': (second_party.isin(['Cars', 'Large motorized vehicle',
                                           'Light motorized vehicle'])
                        & is_driver_or_passenger),
        'MMV': ((second_party == 'Micromobility vehicle')
                & is_driver_or_passenger & (data['severity'] != 3)),
        'Pedestrian': data['Num_Acc'].isin(
            data.loc[second_party == 'Pedestrian', 'Num_Acc']),
        'Single-vehicle': ((second_party == 'No other vehicle')
                           & is_driver_or_passenger),
    }



## 8. Une seule table pour les quatre segments

Les tables de la section 7 decrivent la severite a l'interieur de chaque segment.
Celle-ci met les quatre cote a cote : un groupe de colonnes par segment,
subdivise en NI / I / F (aucune blessure, blessure, deces). Effectifs seuls --
pas de pourcentages, pas de p-values -- pour que treize colonnes tiennent sur
une page. Les variables continues sont resumees par leur mediane, un
median [Q1--Q3] etant illisible sur douze colonnes.


In [10]:
# --- One compact table for the four segments ----------------------------------
# The per-segment tables above answer "how does severity vary inside a segment".
# This one puts the four of them side by side: one group of columns per segment,
# split into NI / I / F (no injury, injury, fatality). Counts only -- no
# percentages, no p-values -- so that thirteen columns still fit on a page.
#
# Variable order and labels are those of `latex_tables.DEFAULT_STRUCTURE` and
# `DEFAULT_CONTINUOUS`, so this table reads like the pooled one. Continuous
# variables are summarised by their median alone: a median [Q1--Q3] in twelve
# columns would not fit.
#
# The MMV segment has no fatality (its single fatal observation is dropped before
# estimation), so its F column stays empty.
#
# Variables describing the crash itself are counted once per accident (asterisk
# in the table), the others once per individual. Second-party characteristics are
# read as "at least one of the other parties has this characteristic" (columns
# `_2` and `_3` combined), so the percentages of one variable may add up to more
# than the sample size.

from latex_tables import (DEFAULT_CONTINUOUS, DEFAULT_STRUCTURE, _escape, _row)

SEGMENT_ORDER = ['Car crashes', 'MMV', 'Pedestrian', 'Single-vehicle']
SEVERITY_COLUMNS = [(1, 'NI'), (2, 'I'), (3, 'F')]


def build_segment_summary(data, segments=None):
    """{(Variable, Category): {segment: {severite: valeur}}} + les effectifs."""
    segments = segments or segment_masks(data)
    samples = {name: data.loc[mask] for name, mask in segments.items()}
    accidents = {name: sample.drop_duplicates('Num_Acc')
                 for name, sample in samples.items()}

    cells, sizes = {}, {}
    for name, sample in samples.items():
        sizes[name] = {severity: int((sample['severity'] == severity).sum())
                       for severity, _ in SEVERITY_COLUMNS}

        for variable in categorical_vars:
            if variable in SECOND_PARTY_GROUPS:
                columns = SECOND_PARTY_GROUPS[variable]
                for level in any_party_levels(sample, columns):
                    mask = any_party_mask(sample, columns, level)
                    if not mask.any():
                        continue
                    counts = sample.loc[mask, 'severity'].value_counts()
                    key = (variable, str(level))
                    for severity, _ in SEVERITY_COLUMNS:
                        count = int(counts.get(severity, 0))
                        if count:
                            cells.setdefault(key, {}).setdefault(
                                name, {})[severity] = f'{count}'
                continue

            frame = accidents[name] if variable in var_for_acc else sample
            frame = frame.dropna(subset=[variable])
            if frame.empty:
                continue
            counts = frame.groupby([variable, 'severity']).size()
            for (category, severity), count in counts.items():
                key = (variable, str(category))
                cells.setdefault(key, {}).setdefault(name, {})[int(severity)] = (
                    f'{int(count)}')

        for variable in continuous_vars:
            frame = accidents[name] if variable in var_for_acc else sample
            frame = frame[['severity', variable]].dropna(subset=[variable])
            frame = frame[frame[variable] != 999]   # sentinelle de donnee manquante
            if frame.empty:
                continue
            # Mediane brute : le nombre de decimales est celui de
            # DEFAULT_CONTINUOUS, applique au moment d'ecrire le tableau.
            medians = frame.groupby('severity')[variable].median()
            key = (variable, '')
            for severity, median in medians.items():
                cells.setdefault(key, {}).setdefault(name, {})[int(severity)] = median

    return cells, sizes


# Les renommages de la section 4, appliques aux cles plutot qu'au DataFrame.
SUMMARY_RENAMING = {
    'vehicle_type_2': 'Third-party vehicle type',
    'Maneuver_2': 'Third-party maneuver',
    'Gender_2': 'Third-party gender',
    'Point of impact_opposite': 'Third-party impact location',
    'vma': 'Speed limit',
    'surfacechaussee': 'Road surface width',
    'pentemoyenne': 'Average slope',
    'largeurtrottoirdroit': 'Sidewalk width',
    'age': 'Individual age',
    'number of involved vehicles': 'Number of vehicles involved',
}


def make_segment_summary_table(
    cells, sizes, segments=None,
    caption=('Composition of the four estimation samples by severity '
             '(NI: no injury, I: injury, F: fatality). Counts for the '
             'categorical variables, median for the continuous ones. '
             'Second-party characteristics count the individuals facing at '
             'least one party with that characteristic.'),
    label='tab:data_segments',
):
    """Longtable : une ligne par modalite, trois colonnes par segment."""
    segments = segments or [name for name in SEGMENT_ORDER if name in sizes]
    renamed = {(SUMMARY_RENAMING.get(variable, variable), category): values
               for (variable, category), values in cells.items()}

    def cells_of(values, formatter=str):
        row = []
        for name in segments:
            by_severity = values.get(name, {})
            for severity, _ in SEVERITY_COLUMNS:
                value = by_severity.get(severity)
                row.append('' if value is None else formatter(value))
        return row

    def line(label_text, values, bold=False, escape=True, formatter=str):
        # Les libelles de DEFAULT_CONTINUOUS sont deja ecrits en LaTeX (\%),
        # les echapper une seconde fois les casserait.
        text = _escape(label_text) if escape else label_text
        first = r'\textbf{' + text + '}' if bold else text
        return _row([first] + cells_of(values, formatter))

    width = len(segments) * len(SEVERITY_COLUMNS) + 1
    span = lambda content: rf'\multicolumn{{{width}}}{{l}}{{' + content + r'} \\'

    # `{|c}` reproduit dans l'en-tete le filet vertical du preambule.
    group_header = _row([''] + [r'\multicolumn{' + str(len(SEVERITY_COLUMNS))
                                + r'}{|c}{\textbf{' + _escape(name) + '}}'
                                for name in segments])
    rules = ''.join(
        rf'\cline{{{2 + position * len(SEVERITY_COLUMNS)}-'
        rf'{1 + (position + 1) * len(SEVERITY_COLUMNS)}}}'
        for position in range(len(segments)))
    severity_header = _row([r'\textbf{Category}']
                           + [r'\textbf{' + short + '}'
                              for _, short in SEVERITY_COLUMNS] * len(segments))
    header = [group_header, rules, severity_header]

    # Un filet vertical avant chaque groupe de colonnes, pour separer les segments.
    columns = 'p{4.2cm}' + '|' + '|'.join(['r' * len(SEVERITY_COLUMNS)] * len(segments))

    lines = [
        r'\renewcommand{\arraystretch}{0.95}',
        r'\setlength{\tabcolsep}{3pt}',
        r'\small',
        rf'\begin{{longtable}}{{{columns}}}',
        rf'\caption{{{caption}}} \label{{{label}}} \\',
        r'\hline', *header, r'\hline', r'\endfirsthead',
        span(r'\textit{(continued)}'),
        r'\hline', *header, r'\hline', r'\endhead',
        r'\hline', span(r'\textit{Continued on next page}'), r'\endfoot',
        r'\hline', r'\endlastfoot',
        line('Observations', {name: sizes[name] for name in segments}, bold=True),
        r'\hline',
    ]

    for section_title, variables in DEFAULT_STRUCTURE:
        section = []
        for variable, display_label, categories in variables:
            present = {category: values
                       for (name, category), values in renamed.items()
                       if name == variable}
            if not present:
                continue

            order = {str(key): value for key, value in (categories or {}).items()}
            for raw in sorted(present):
                order.setdefault(raw, raw)

            rows = [line(shown, present[raw]) for raw, shown in order.items()
                    if raw in present]
            if rows:
                star = '$^{*}$' if variable in CRASH_LEVEL_STARRED else ''
                section.append(span('{' + _escape(display_label) + star + '}'))
                section.extend(rows)
                section.append(r'\hline')

        if section:
            lines.append(span(r'\textbf{' + _escape(section_title) + '}'))
            lines.append(r'\hline')
            lines.extend(section)

    continuous = [(variable, display_label, decimals)
                  for variable, display_label, decimals in DEFAULT_CONTINUOUS
                  if (variable, '') in renamed]
    if continuous:
        lines.append(span(r'\textbf{Continuous variables (median)}'))
        lines.append(r'\hline')
        for variable, display_label, decimals in continuous:
            lines.append(line(display_label, renamed[(variable, '')],
                              escape=False,
                              formatter=lambda value, d=decimals: f'{value:.{d}f}'))
        lines.append(r'\hline')

    lines.append(span(CRASH_LEVEL_NOTE))
    lines.append(r'\end{longtable}')
    return '\n'.join(lines)


summary_cells, summary_sizes = build_segment_summary(dataset)
segment_summary_tex = make_segment_summary_table(summary_cells, summary_sizes)

summary_path = SEGMENT_TABLES_DIRECTORY / 'table_descriptive_segments.tex'
summary_path.write_text(segment_summary_tex + '\n', encoding='utf-8')
print(summary_path)
print()
print(segment_summary_tex[:2200])

results/tables/table_descriptive_segments.tex

\renewcommand{\arraystretch}{0.95}
\setlength{\tabcolsep}{3pt}
\small
\begin{longtable}{p{4.2cm}|rrr|rrr|rrr|rrr}
\caption{Composition of the four estimation samples by severity (NI: no injury, I: injury, F: fatality). Counts for the categorical variables, median for the continuous ones. Second-party characteristics count the individuals facing at least one party with that characteristic.} \label{tab:data_segments} \\
\hline
 & \multicolumn{3}{|c}{\textbf{Car crashes}} & \multicolumn{3}{|c}{\textbf{MMV}} & \multicolumn{3}{|c}{\textbf{Pedestrian}} & \multicolumn{3}{|c}{\textbf{Single-vehicle}} \\
\cline{2-4}\cline{5-7}\cline{8-10}\cline{11-13}
\textbf{Category} & \textbf{NI} & \textbf{I} & \textbf{F} & \textbf{NI} & \textbf{I} & \textbf{F} & \textbf{NI} & \textbf{I} & \textbf{F} & \textbf{NI} & \textbf{I} & \textbf{F} \\
\hline
\endfirsthead
\multicolumn{13}{l}{\textit{(continued)}} \\
\hline
 & \multicolumn{3}{|c}{\textbf{Car crashes}} & \

## 9. Annexe : dictionnaire des variables

Le tableau d'annexe du manuscrit, rempli depuis les donnees plutot qu'a la main :
une ligne par variable avec son explication et sa source, puis une ligne indentee
par modalite. Moyenne (ecart-type) pour les continues, pourcentage par modalite
pour les categorielles, part de valeurs manquantes en derniere colonne. Ecrit
dans `results/tables/descriptive_appendix.tex`.

Les sources (`Police report`, `GIS`, `Derived`) sont a verifier une par une
contre le chapitre de collecte des donnees.


In [11]:
# --- Appendix: variable dictionary + univariate statistics --------------------
# The appendix table of the manuscript, filled from the data instead of by hand:
# one row per variable with its explanation and its source, then one indented row
# per level. Values are computed on the estimation sample (`dataset`, the union
# of the four segments): mean (SD) for continuous variables, percentage per level
# for categorical ones, and the share of missing values when there is any.
#
# `POLICE` / `GIS` / `DERIVED` are the sources as worded in the manuscript.
# Check each assignment against your data-collection chapter: only you know
# which layer every geometric variable came from.

APPENDIX_PATH = SEGMENT_TABLES_DIRECTORY / 'descriptive_appendix.tex'

# La colonne « Source » n'est PAS echappee : elle accepte du LaTeX, donc des
# citations. Deux precautions :
#   - prefixe `r` pour que Python ne lise pas `\c` comme une sequence
#     d'echappement (SyntaxWarning en 3.12+, et source de bugs silencieux) ;
#   - une seule commande \citep a plusieurs cles, et non deux \citep colles :
#     natbib rend « (A ; B) » au lieu de « (A)(B) ».
POLICE = 'Police report'
GIS = 'GIS'
GIS_POLICE = 'GIS / Police report'
DERIVED = 'Derived'

LYON = (r'\citep{metropoledelyonChausseesTrottoirsMetropole2025,'
        r'metropoledelyonAmenagementsCyclablesMetropole2025}')
PARIS = r'\citep{regioniledefranceAmenagementsVeloIledeFrance2024}'
LYON_PARIS = (r'\citep{metropoledelyonChausseesTrottoirsMetropole2025,'
              r'metropoledelyonAmenagementsCyclablesMetropole2025,'
              r'regioniledefranceAmenagementsVeloIledeFrance2024}')
# Prete a l'emploi dans APPENDIX_STRUCTURE : « GIS (Lyon ; Paris) ».
GIS_CITED = 'GIS ' + LYON_PARIS

# (section, [(colonne, libelle, explication, source, ordre des modalites)])
APPENDIX_STRUCTURE = [
    ('Dependent variable', [
        ('severity', 'Injury severity', 'Injury severity level', POLICE,
         {1: 'No injury', 2: 'Injury', 3: 'Fatality'}),
    ]),
    ('Individual and vehicle characteristics', [
        ('age', 'Age (years)', 'Age of the involved road user', POLICE, None),
        ('Gender', 'Gender', 'Sex of the involved road user', POLICE, None),
        ('User category', 'User category', 'Role of the individual in the crash',
         POLICE, None),
        ('Vehicle', 'Vehicle', 'Type of vehicle used', POLICE, None),
        ('Helmet', 'Helmet', 'Helmet use', POLICE, None),
        ('Reflective jacket', 'Reflective jacket', 'Reflective jacket use',
         POLICE, None),
        ('Number of passengers', 'Number of passengers',
         'Number of passengers on the vehicle', POLICE, None),
        ('Trip purpose', 'Trip purpose', 'Reason for the trip', POLICE, None),
    ]),
    ('Crash characteristics', [
        ('number of involved vehicles', 'Number of involved vehicles',
         'Total number of vehicles involved', POLICE, None),
        ('Point of impact', 'Point of impact', 'Main collision point', POLICE,
         None),
        ('Maneuver', 'Maneuver (individual)',
         'Maneuver performed before the crash', POLICE, None),
        ('Surface condition', 'Surface condition', 'Road surface condition',
         POLICE, None),
        ('Accident location', 'Crash location',
         'Part of the road space where the crash occurred', POLICE, None),
    ]),
    ('Second-party characteristics', [
        ('vehicle_type_2', 'Second-party vehicle',
         'Vehicle type of the collision partner', POLICE, None),
        ('Maneuver_2', 'Second-party maneuver',
         'Maneuver of the collision partner', POLICE, None),
        ('Gender_2', 'Second-party gender', 'Sex of the collision partner',
         POLICE, None),
        ('age_opposite_mean', 'Age opposite (years)', 'Mean age of the collision partners; missing when no second party is identified', POLICE, None),
    ]),
    ('Infrastructure and environment', [
        ('Cycle facilities', 'Cycle facility',
         'Presence of cycling infrastructure', POLICE, None),
        ('Intersection', 'Intersection', 'Crash occurred at an intersection',
         POLICE, None),
        ('Crossroad', 'Signalized intersection', 'Presence of traffic signals',
         GIS, None),
        ('Lighting conditions', 'Lighting conditions',
         'Ambient lighting at crash location', POLICE, None),
        ('Weather conditions', 'Weather conditions',
         'Atmospheric conditions at the time of the crash', POLICE, None),
        ('Long profile', 'Slope', 'Presence of road gradient', POLICE, None),
        ('Pavement', 'Pavement', 'Surface material of the roadway', LYON, None),
        ('vma', 'Posted speed limit (km/h)', 'Legal speed limit', POLICE, None),
        ('largeurchaussee', 'Road width (m)', 'Width of the roadway', LYON, None),
        ('pentemoyenne', 'Average slope (%)',
         'Mean longitudinal slope of the road section', GIS, None),
        ('largeurtrottoirdroit', 'Sidewalk width (m)',
         'Width of the right-hand sidewalk', LYON, None),
        ('Agglomeration', 'Metropolitan area',
         'Metropolitan area the crash belongs to', POLICE, None),
    ]),
]


def _appendix_entries(data, accidents):
    """Par section : (libelle, explication, source, statistique, manquants, modalites)."""
    for section, variables in APPENDIX_STRUCTURE:
        entries = []
        for column, label, explanation, source, ordering in variables:
            if column in SECOND_PARTY_GROUPS:
                columns = SECOND_PARTY_GROUPS[column]
                levels = [(str(level),
                           100 * float(any_party_mask(data, columns, level).mean()))
                          for level in any_party_levels(data, columns)]
                present = pd.concat([data[c].notna() for c in columns
                                     if c in data.columns], axis=1).any(axis=1)
                missing = 100 * float((~present).mean())
                entries.append((label, explanation, source, '', missing, levels,
                                column in var_for_acc))
                continue

            frame = accidents if column in var_for_acc else data
            if column not in frame.columns:
                print(f'  [{column}] absente du jeu de donnees, ignoree')
                continue

            values = frame[column]
            missing = float(values.isna().mean() * 100)
            values = values.dropna()
            if values.empty:
                continue

            # Une colonne numerique a beaucoup de valeurs distinctes est
            # continue, meme si elle n'est pas dans `continuous_vars` :
            # `age_opposite_mean` sinon produirait une modalite par age.
            is_continuous = ordering is None and (
                column in continuous_vars
                or (values.dtype.kind in 'fi' and values.nunique() > 12))
            if is_continuous:
                numeric = values[values != 999]   # sentinelle de donnee manquante
                statistic = f'{numeric.mean():.1f} ({numeric.std():.1f})'
                entries.append((label, explanation, source, statistic, missing,
                                [], column in var_for_acc))
            else:
                shares = values.value_counts(normalize=True) * 100
                if ordering:
                    levels = [(ordering[key], shares.get(key, 0.0))
                              for key in ordering if key in shares.index]
                else:
                    levels = [(str(key), share) for key, share
                              in shares.sort_values(ascending=False).items()]
                entries.append((label, explanation, source, '', missing, levels,
                                column in var_for_acc))
        if entries:
            yield section, entries


def make_appendix_table(
    data,
    caption=('Univariate descriptive statistics on the estimation sample '
             '($N = {n}$). For continuous variables, mean and standard '
             'deviation are reported; for categorical variables, the percentage '
             'of observations in each level is reported.'),
    label='tab:descriptive_appendix',
):
    accidents = data.drop_duplicates('Num_Acc')
    header = [_row([r'\textbf{Variable}', r'\textbf{Explanation}',
                    r'\textbf{Source}', r'\textbf{Mean (SD) / \%}',
                    r'\textbf{$N$ / missing}'])]

    lines = [
        r'\begin{longtable}{p{3cm} p{3cm} p{2.5cm} p{2.8cm} p{2.5cm}}',
        rf'\caption{{{caption.format(n=f"{len(data):,}".replace(",", "{,}"))}}}',
        rf'\label{{{label}}} \\',
        r'\hline', *header, r'\hline', r'\endfirsthead',
        '',
        r'\multicolumn{5}{l}{\textit{(continued)}} \\',
        r'\hline', *header, r'\hline', r'\endhead',
        '',
        r'\hline',
        r'\multicolumn{5}{r}{\textit{Continued on next page}} \\',
        r'\endfoot',
        '',
        r'\hline', r'\endlastfoot',
        '',
    ]

    for section, entries in _appendix_entries(data, accidents):
        lines.append(r'\multicolumn{5}{c}{\textbf{' + _escape(section) + r'}} \\')
        lines.append(r'\hline')
        lines.append('')
        for (label_text, explanation, source, statistic, missing, levels,
             crash_level) in entries:
            missing_cell = f'missing: {missing:.1f}\\%' if missing >= 0.05 else ''
            # `source` n'est pas echappe : il peut contenir un \citep ;
            # l'asterisque est ajoute apres l'echappement du libelle.
            star = r'$^{*}$' if crash_level else ''
            lines.append(_row([_escape(label_text) + star, _escape(explanation),
                               source, statistic, missing_cell]))
            for level, share in levels:
                lines.append(_row([r'\quad ' + _escape(str(level)), '', '',
                                   f'{share:.1f}\\%', '']))
            lines.append('')
        lines.append(r'\hline')
        lines.append('')

    lines += [
        r'\multicolumn{5}{p{\linewidth}}{\footnotesize Continuous variables: '
        r'mean (standard deviation). Categorical variables: percentage of '
        r'observations per level, computed on the non-missing observations. '
        r'Source indicates the origin of each variable (police crash report, '
        r'GIS, or derived variable). $^{*}$ marks a variable describing the '
        r'crash itself, counted once per accident; the others are counted once '
        r'per individual. Second-party rows report the share of individuals '
        r'facing at least one party with that characteristic, so they may add '
        r'up to more than 100\%.} \\',
        r'\end{longtable}',
    ]
    return '\n'.join(lines)


appendix_tex = make_appendix_table(dataset)
APPENDIX_PATH.write_text(appendix_tex + '\n', encoding='utf-8')
print(APPENDIX_PATH)
print()
print('\n'.join(appendix_tex.splitlines()[:45]))

results/tables/descriptive_appendix.tex

\begin{longtable}{p{3cm} p{3cm} p{2.5cm} p{2.8cm} p{2.5cm}}
\caption{Univariate descriptive statistics on the estimation sample ($N = 15{,}386$). For continuous variables, mean and standard deviation are reported; for categorical variables, the percentage of observations in each level is reported.}
\label{tab:descriptive_appendix} \\
\hline
\textbf{Variable} & \textbf{Explanation} & \textbf{Source} & \textbf{Mean (SD) / \%} & \textbf{$N$ / missing} \\
\hline
\endfirsthead

\multicolumn{5}{l}{\textit{(continued)}} \\
\hline
\textbf{Variable} & \textbf{Explanation} & \textbf{Source} & \textbf{Mean (SD) / \%} & \textbf{$N$ / missing} \\
\hline
\endhead

\hline
\multicolumn{5}{r}{\textit{Continued on next page}} \\
\endfoot

\hline
\endlastfoot

\multicolumn{5}{c}{\textbf{Dependent variable}} \\
\hline

Injury severity & Injury severity level & Police report &  &  \\
\quad No injury &  &  & 12.9\% &  \\
\quad Injury &  &  & 86.5\% &  \\
\quad Fatali